# Steps for Colab


1.   Add mistral-asl-instruct.zip provided from Mistral_Instruct_WorkingTrainer.ipynb
2.   Run cells until huggingface login() prompt, and provide your authentication token (recommended, but can be skipped)
3.   Run remaining cells. Last cell will allow you to test the fine-tuned model

In [ ]:
!pip install bitsandbytes

# Login with Hugging Face API for authenticated access to the base model. Not necessary, but recommended for more bandwidth

In [ ]:
import os
from huggingface_hub import login, get_token

# Check if API token is already stored
if get_token() is None:
    login()
else:
    print("Already logged into Hugging Face!")

# Unpack .zip file provided by Mistral-Instruct-WorkingTrainer.ipynb

In [2]:
import zipfile
import os

zip_path = "/content/mistral-asl-instruct.zip"
extract_dir = "mistral-asl-instruct"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Load base model and apply quantized LoRA weights for finetuning (Hugging Face access recommended)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
import torch

# Load PEFT-config
peft_config = PeftConfig.from_pretrained("./mistral-asl-instruct")

# 4-bit quantizing
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    peft_config.base_model_name_or_path,
    device_map="cuda:0",
    trust_remote_code=True,
    quantization_config=quant_config
)

# Apply LoRA weights for finetuning
model = PeftModel.from_pretrained(base_model, "./mistral-asl-instruct")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("./mistral-asl-instruct", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# TESTING: Test a list of prompts. The following cell allows you to test a single prompt

In [ ]:
# test inference of the model with a list of prompts

prompts = list()
prompts.append("Could you loan me five dollars?")
prompts.append("Did you find the ASL book?")
prompts.append("Guess how old I am!")
prompts.append("If a student cheats on a test, do you think they should be expelled?")
prompts.append("Mommy loves you!")
prompts.append('Show me a sentence that uses the sign "WILL".')
prompts.append("What are you doing next weekend?")
prompts.append("What color is a stop sign?")
prompts.append("What did you eat this morning?")
prompts.append("I had pizza last night.")

for prompt_text in prompts:
  messages = [
    {"role": "user", "content": "Translate the following to American Sign Language (ASL) structure:\n" + prompt_text}
  ]

  # Step 1: Apply chat template to get the formatted string
  formatted_chat_string = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

  # Step 2: Tokenize the formatted string to get input_ids and attention_mask as a BatchEncoding object
  inputs = tokenizer(formatted_chat_string, return_tensors="pt").to(model.device)

  # Step 3: Pass input_ids and attention_mask explicitly to model.generate
  output = model.generate(
      input_ids=inputs.input_ids,
      attention_mask=inputs.attention_mask,
      max_new_tokens=64,
      pad_token_id=tokenizer.eos_token_id
  )

  result = tokenizer.decode(output[0], skip_special_tokens=True)

  result = result.replace("[INST]","")
  result = result.replace("[/INST]","\n")
  print("\n\n" + result)

In [ ]:
prompt = "What color is your shirt?"

messages = [
    {"role": "user", "content": "Translate the following to American Sign Language (ASL) structure:\n" + prompt}
]

# Step 1: Apply chat template to get the formatted string
formatted_chat_string = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Step 2: Tokenize the formatted string to get input_ids and attention_mask as a BatchEncoding object
inputs = tokenizer(formatted_chat_string, return_tensors="pt").to(model.device)

# Step 3: Pass input_ids and attention_mask explicitly to model.generate
output = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=64,
    pad_token_id=tokenizer.eos_token_id
)

result = tokenizer.decode(output[0], skip_special_tokens=True)

result = result.replace("[INST]","\n")
result = result.replace("[/INST]","\n")
print("\n\n" + result)